In [4]:
import os
from PIL import Image

# ===========================================================================
# CONFIGURATION - CHANGE YOUR PATHS HERE
# ===========================================================================
# Hint: On Windows, use forward slashes / or prefix with an r (e.g., r"C:\Images")
INPUT_DIR   = r"D:\\Monash\\Year3\\ECE3073\\miniproj_m2\\software"
OUTPUT_DIR  = r"D:\\Monash\\Year3\\ECE3073\\miniproj_m2\\software\\core2_rtos_bsp"
IMAGE_FILE  = "emergencyIMG2.jpg"  # The image you want to convert

TARGET_W = 320
TARGET_H = 240

# ===========================================================================
# CONVERSION ENGINE
# ===========================================================================
def rgb888_to_rgb332(r, g, b):
    r3 = (r >> 5) & 0x07
    g3 = (g >> 5) & 0x07
    b2 = (b >> 6) & 0x03
    return (r3 << 5) | (g3 << 2) | b2

def convert_notebook():
    input_path = os.path.join(INPUT_DIR, IMAGE_FILE)
    
    # Check if the input file actually exists
    if not os.path.exists(input_path):
        print(f"ERROR: Could not find the input image at:\n  --> {input_path}")
        print("Please check your INPUT_DIR or IMAGE_FILE spelling.")
        return

    # Auto-generate the output .h filename based on the image name
    base_name = os.path.splitext(IMAGE_FILE)[0]
    output_path = os.path.join(OUTPUT_DIR, base_name + ".h")

    # Create the output directory if it doesn't exist yet
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    # Open and resize with letterbox onto black 320x240 canvas
    img = Image.open(input_path).convert("RGB")
    print(f"Successfully Opened: {IMAGE_FILE} ({img.width}x{img.height})")

    img_ratio    = img.width / img.height
    target_ratio = TARGET_W / TARGET_H
    if img_ratio > target_ratio:
        new_w, new_h = TARGET_W, int(TARGET_W / img_ratio)
    else:
        new_w, new_h = int(TARGET_H * img_ratio), TARGET_H

    img    = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (TARGET_W, TARGET_H), (0, 0, 0))
    canvas.paste(img, ((TARGET_W - new_w) // 2, (TARGET_H - new_h) // 2))

    # Convert to RGB332
    pixels = list(canvas.getdata())
    rgb332 = [rgb888_to_rgb332(r, g, b) for r, g, b in pixels]

    # Clean variable name for C array syntax
    var_name = base_name.replace("-","_").replace(" ","_")

    # Write header file
    with open(output_path, "w") as f:
        f.write(f"// Auto-generated by img_to_rgb332.py\n")
        f.write(f"// Source : {IMAGE_FILE}\n")
        f.write(f"// Format : RGB332 (8-bit/pixel), {TARGET_W}x{TARGET_H}\n")
        f.write(f"// Layout : R[7:5] G[4:2] B[1:0]\n\n")
        f.write(f"#ifndef {var_name.upper()}_H\n")
        f.write(f"#define {var_name.upper()}_H\n\n")
        f.write(f"#include <stdint.h>\n\n")
        f.write(f"static const uint8_t {var_name}[{len(rgb332)}] = {{\n")
        
        COLS = 16
        for i, val in enumerate(rgb332):
            if i % COLS == 0:
                f.write("    ")
            f.write(f"0x{val:02X}")
            if i < len(rgb332) - 1:
                f.write(",")
            if i % COLS == COLS - 1:
                f.write("\n")
        f.write("\n};\n\n")
        f.write(f"#endif // {var_name.upper()}_H\n")

    print(f"Done! --> Saved header file to: {output_path}")
    print(f"C Array Generated: static const uint8_t {var_name}[{len(rgb332)}]")

# Run the converter
convert_notebook()

Successfully Opened: emergencyIMG2.jpg (491x350)
Done! --> Saved header file to: D:\\Monash\\Year3\\ECE3073\\miniproj_m2\\software\\core2_rtos_bsp\emergencyIMG2.h
C Array Generated: static const uint8_t emergencyIMG2[76800]
